# Step 2: Baseline XGBoost Eligibility Model

This notebook trains the baseline classifier that every later audit module (fairness, explainability, reliability, robustness) will examine.

The goal here is not to tune the model. It is to get a reasonable, reproducible model and to save its predictions and probabilities in a form the audit modules can reuse.

Steps:
1. Load the processed split from notebook 01
2. Prepare features (native categorical handling for high-cardinality columns)
3. Train XGBoost
4. Evaluate overall performance (accuracy, precision, recall, F1, ROC-AUC)
5. Quick per-group performance check (a preview of what the Fairness module formalises)
6. Save the model and predictions

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from src.model import (
    load_processed, to_model_frame, train_baseline, evaluate, FEATURE_COLS,
)

PROCESSED_DIR = os.path.abspath("../data/processed")
RESULTS_DIR = os.path.abspath("../results")
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. Load the processed split

`train.csv` and `test.csv` were written by notebook 01 after de-duplication and the stratified 80/20 split. Each file holds the 10 features plus `label` (income > $50,000, the eligibility proxy) and `group` (RAC1P).

In [2]:
X_train, X_test, y_train, y_test, g_train, g_test = load_processed(PROCESSED_DIR)
print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Train positive rate:", round(y_train.mean(), 4), " Test positive rate:", round(y_test.mean(), 4))

X_train: (154566, 10)  X_test: (38642, 10)
Train positive rate: 0.4789  Test positive rate: 0.479


## 2. Feature preparation

Notebook 01 flagged `OCCP` (528 codes) and `POBP` (221 codes) as high-cardinality. One-hot encoding them would create hundreds of sparse columns and would also fragment SHAP importance across dummy columns later.

Instead, the 8 code-like columns (`COW, SCHL, MAR, OCCP, POBP, RELSHIPP, SEX, RAC1P`) are cast to pandas `category` dtype and XGBoost's native categorical support (`enable_categorical=True`) is used. `AGEP` and `WKHP` stay numeric.

The category sets are built from the union of train and test so both splits share identical category codes.

In [3]:
Xtr, Xte = to_model_frame(X_train, X_test)
print(Xtr.dtypes)

AGEP         float64
WKHP         float64
COW         category
SCHL        category
MAR         category
OCCP        category
POBP        category
RELSHIPP    category
SEX         category
RAC1P       category
dtype: object


## 3. Train XGBoost

Fixed, moderate hyperparameters (300 trees, depth 6, learning rate 0.1, `hist` tree method, seed 42). This is a baseline for auditing, so no hyperparameter search is done. Keeping it fixed also keeps every later audit result reproducible.

In [4]:
model = train_baseline(Xtr, y_train)
model

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## 4. Overall performance

Predictions use a 0.5 probability threshold. The probabilities themselves (`y_proba`) are kept because the Reliability module needs them for the Brier Score and Expected Calibration Error.

In [5]:
y_proba = model.predict_proba(Xte)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

metrics = evaluate(y_test, y_pred, y_proba)
pd.Series(metrics).round(4).rename("test_set").to_frame()

,test_set
accuracy,0.8156
precision,0.8040
recall,0.8131
f1,0.8085
roc_auc,0.8963


## 5. Per-group preview

A quick look at accuracy and the selection rate (share predicted eligible) per RAC1P group and per SEX group. This is only a preview. The Fairness module will compute Demographic Parity, Disparate Impact and Equal Opportunity formally.

Group `n` is shown beside every number because small groups (RAC1P = 4 has only a handful of rows) give unstable estimates.

In [6]:
preds = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": y_pred,
    "y_proba": y_proba,
    "RAC1P": X_test["RAC1P"].astype(int).values,
    "SEX": X_test["SEX"].astype(int).values,
})
preds["correct"] = (preds.y_true == preds.y_pred).astype(int)

def group_preview(col):
    return preds.groupby(col).agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        selection_rate=("y_pred", "mean"),
        true_rate=("y_true", "mean"),
    ).round(4)

print("By RAC1P:")
print(group_preview("RAC1P"))
print("\nBy SEX:")
print(group_preview("SEX"))

By RAC1P:
           n  accuracy  selection_rate  true_rate
RAC1P                                            
1      15903    0.8122          0.6152     0.5867
2       1637    0.7666          0.4258     0.4319
3        502    0.7789          0.3108     0.3446
4          1    1.0000          1.0000     1.0000
5         84    0.8095          0.2976     0.2976
6       7189    0.8185          0.5606     0.5432
7        139    0.7482          0.3669     0.4173
8       6706    0.8275          0.2273     0.2692
9       6481    0.8249          0.3780     0.3862

By SEX:
         n  accuracy  selection_rate  true_rate
SEX                                            
1    20370    0.8136          0.5336     0.5239
2    18272    0.8178          0.4295     0.4289


**How to read this:** compare `selection_rate` with `true_rate` within each group. If the model's selection rate is much further from the true rate for one group than for others, that is an early sign of group-dependent behaviour worth the formal fairness audit.

## 6. Save the model and predictions

The audit modules read from these files instead of retraining, so every module audits the exact same model and predictions.

In [7]:
model.save_model(os.path.join(RESULTS_DIR, "baseline_xgb.json"))
preds.drop(columns="correct").to_csv(os.path.join(RESULTS_DIR, "baseline_predictions.csv"), index=False)
pd.Series(metrics).to_json(os.path.join(RESULTS_DIR, "baseline_metrics.json"), indent=2)
print("Saved model, predictions and metrics to", RESULTS_DIR)

Saved model, predictions and metrics to E:\MTECH\Final_yr_project\results


## Summary and next step

- Trained a fixed-hyperparameter XGBoost baseline using native categorical handling.
- Reported accuracy, precision, recall, F1 and ROC-AUC on the held-out test set.
- Saved the model, test predictions (with probabilities and protected attributes) and metrics to `results/`.

**Next:** notebook 03, the Fairness audit module (Demographic Parity Difference, Disparate Impact Ratio, Equal Opportunity Difference) on `results/baseline_predictions.csv`.